# CDR-MLC routing diagnostics

This notebook diagnoses why samples from a different congestion condition are routed to an expert whose training distribution may not match them.

**Leakage boundary:** K-Means routing and expert training use training data only. Test labels are used only after prediction for diagnostic tables (accuracy, class distribution, and confusion matrix); they never change clusters, thresholds, or predictions.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    display
except NameError:
    display = print

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
RANDOM_STATE = 42
BASE = Path('DATASETS/CDR-MLC/scale_1')
SCENARIOS = {
    'scenario_1': (BASE/'Short/level_1.csv', BASE/'Short/level_2.csv'),
    'scenario_2': (BASE/'Short/level_1.csv', BASE/'Short/level_3.csv'),
    'scenario_3': (BASE/'Short/level_2.csv', BASE/'Short/level_3.csv'),
    'scenario_4': (BASE/'Short/CDR-MLC-Shuffle.csv', BASE/'Long/CDR-MLC-Shuffle.csv'),
    'scenario_5': (BASE/'Long/CDR-MLC-Shuffle.csv', BASE/'Short/CDR-MLC-Shuffle.csv'),
}

# Load the exact leakage-safe implementation from the first cell of CDR-MLC.ipynb.
main_notebook = json.loads(Path('CDR-MLC.ipynb').read_text(encoding='utf-8'))
core_source = ''.join(main_notebook['cells'][0]['source'])
exec(compile(core_source, 'CDR-MLC.ipynb::core', 'exec'), globals())
print('Loaded the exact routing and expert implementation from CDR-MLC.ipynb')


In [ ]:
def _domain_auc(train_part, test_part, features, max_per_domain=3000):
    """How easily feature values reveal train vs test domain (0.5=same, 1=different)."""
    if len(train_part) < 10 or len(test_part) < 10:
        return np.nan
    n = min(max_per_domain, len(train_part), len(test_part))
    a = train_part[features].sample(n=n, random_state=RANDOM_STATE)
    b = test_part[features].sample(n=n, random_state=RANDOM_STATE)
    X = pd.concat([a, b], ignore_index=True).replace([np.inf, -np.inf], np.nan).fillna(0)
    y = np.r_[np.zeros(n), np.ones(n)]
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
    )
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model.fit(X_train, y_train)
    return float(roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1]))


def _feature_shift_table(train_part, test_part, features):
    rows = []
    eps = 1e-12
    for feature in features:
        a = train_part[feature].dropna().to_numpy(dtype=float)
        b = test_part[feature].dropna().to_numpy(dtype=float)
        if len(a) == 0 or len(b) == 0:
            continue
        train_std = np.std(a, ddof=1)
        train_iqr = np.quantile(a, .75) - np.quantile(a, .25)
        rows.append({
            'feature': feature,
            'train_mean': np.mean(a),
            'test_mean': np.mean(b),
            'standardized_mean_shift': abs(np.mean(b)-np.mean(a))/(train_std+eps),
            'normalized_wasserstein': wasserstein_distance(a,b)/max(train_iqr, train_std, eps),
            'ks_statistic': ks_2samp(a,b).statistic,
        })
    return pd.DataFrame(rows).sort_values(
        ['ks_statistic','normalized_wasserstein'], ascending=False
    ).reset_index(drop=True)


def diagnose_scenario(name, top_features=12):
    train_path, test_path = SCENARIOS[name]
    print(f'\n{name}: {train_path} -> {test_path}')
    result = run_pipeline_from_two_files(
        train_file=str(train_path), test_file=str(test_path),
        n_clusters=3, window_size=3,
        clustering_stats=['mean','median','std','min','max']
    )
    train_df, test_df = result['train_df'], result['test_df']
    features = result['classification_features']
    fixed = result['used_fixed_features']
    stats = result['clustering_stats']

    train_stats, _ = compute_sliding_window_stats(train_df, fixed, result['window_size'], stats)
    test_stats, _ = compute_sliding_window_stats(test_df, fixed, result['window_size'], stats)
    train_scaled = result['scaler'].transform(train_stats)
    test_scaled = result['scaler'].transform(test_stats)
    train_routes = train_df['cluster'].to_numpy()
    test_routes = test_df['cluster'].to_numpy()
    train_distance = result['kmeans_model'].transform(train_scaled)[np.arange(len(train_df)), train_routes]
    test_distance = result['kmeans_model'].transform(test_scaled)[np.arange(len(test_df)), test_routes]

    summary, shifts, label_tables, confusions = [], {}, {}, {}
    class_ids = np.arange(len(result['label_encoder'].classes_))
    for cid in range(3):
        train_mask, test_mask = train_routes == cid, test_routes == cid
        tr, te = train_df.loc[train_mask], test_df.loc[test_mask]
        if len(te) and cid in result['classifiers']:
            pred = result['classifiers'][cid].predict(te[features])
            acc = accuracy_score(te[result['target_column']], pred)
            f1 = f1_score(te[result['target_column']], pred, average='weighted', zero_division=0)
            confusions[cid] = confusion_matrix(te[result['target_column']], pred, labels=class_ids)
        else:
            acc = f1 = np.nan

        train_labels = tr[result['target_column']].value_counts(normalize=True).reindex(class_ids, fill_value=0)
        test_labels = te[result['target_column']].value_counts(normalize=True).reindex(class_ids, fill_value=0)
        label_tables[cid] = pd.DataFrame({
            'class': result['label_encoder'].inverse_transform(class_ids),
            'train_share': train_labels.to_numpy(), 'test_share': test_labels.to_numpy()
        })
        js = float(jensenshannon(train_labels, test_labels, base=2)**2) if len(tr) and len(te) else np.nan
        shift = _feature_shift_table(tr, te, features) if len(tr) and len(te) else pd.DataFrame()
        shifts[cid] = shift
        summary.append({
            'cluster/expert': cid,
            'train_samples': len(tr), 'test_routed_samples': len(te),
            'train_share_%': 100*len(tr)/len(train_df), 'test_share_%': 100*len(te)/len(test_df),
            'expert_accuracy': acc, 'expert_f1_weighted': f1,
            'train_distance_mean': np.mean(train_distance[train_mask]) if train_mask.any() else np.nan,
            'test_distance_mean': np.mean(test_distance[test_mask]) if test_mask.any() else np.nan,
            'distance_mean_ratio': (np.mean(test_distance[test_mask])/(np.mean(train_distance[train_mask])+1e-12)) if train_mask.any() and test_mask.any() else np.nan,
            'label_JS_divergence': js,
            'domain_classifier_AUC': _domain_auc(tr, te, features),
            'mean_feature_KS': shift['ks_statistic'].mean() if len(shift) else np.nan,
        })

    summary = pd.DataFrame(summary)
    print('\nPer-expert routing and transfer summary')
    display(summary.round(4))
    for cid in range(3):
        print(f'\nExpert {cid}: class composition (post-hoc diagnostic only)')
        display(label_tables[cid].round(4))
        print(f'Expert {cid}: strongest shifted classification features')
        display(shifts[cid].head(top_features).round(4))

    route_plot = summary.set_index('cluster/expert')[['train_share_%','test_share_%']]
    route_plot.plot.bar(figsize=(8,4), rot=0, title=f'{name}: routing share, train vs test')
    plt.ylabel('Samples (%)'); plt.grid(axis='y', alpha=.25); plt.tight_layout(); plt.show()

    return {
        'pipeline_result': result, 'summary': summary, 'feature_shifts': shifts,
        'label_distributions': label_tables, 'confusion_matrices': confusions,
        'train_routing_distance': train_distance, 'test_routing_distance': test_distance,
    }


In [ ]:
# Run Scenario 1 independently
scenario_1_diagnostics = diagnose_scenario('scenario_1')


In [ ]:
# Run Scenario 2 independently
scenario_2_diagnostics = diagnose_scenario('scenario_2')


In [ ]:
# Run Scenario 3 independently
scenario_3_diagnostics = diagnose_scenario('scenario_3')


In [ ]:
# Run Scenario 4 independently
scenario_4_diagnostics = diagnose_scenario('scenario_4')


In [ ]:
# Run Scenario 5 independently
scenario_5_diagnostics = diagnose_scenario('scenario_5')
